
# Deep Learning Hyperparameter Tuning — Practical Notebook

This notebook demonstrates how to apply **hyperparameter tuning** methods to the same neural network and the same dataset so the differences between the methods are clear.

We will cover:

1. **Baseline Model**
2. **Grid Search**
3. **Random Search**
4. **Keras Tuner — RandomSearch**
5. **Bayesian Optimization — Keras Tuner**
6. **Hyperband / Successive Halving — Keras Tuner**
7. **Optuna — TPE + Pruning**
8. **Compare Results**
9. **Final Test Evaluation**

> **Main idea:** We do not manually change values trial by trial.  
> We define a **search space** in code, then the search algorithm/tool automatically trains multiple models and compares them using the **validation set**.



## 0. Strategy vs Tool

| Item | Type | What it does |
|---|---|---|
| Grid Search | Search strategy | Tries every combination in a fixed grid |
| Random Search | Search strategy | Samples a limited number of random combinations |
| Bayesian Optimization | Search strategy | Uses previous trials to choose promising next trials |
| Hyperband | Resource-allocation strategy | Stops weak trials early and gives more resources to promising trials |
| Keras Tuner | Tool / library | Implements Random Search, Bayesian Optimization, Hyperband, etc. |
| Optuna | Tool / library | Runs automated optimization using samplers such as TPE and supports pruning |

**Important:**  
`Keras Tuner` and `Optuna` are not hyperparameters, and they are not the same type of concept as `Grid Search`.  
They are tools/libraries that automate the tuning process.



## 1. Install Required Packages

In Google Colab, TensorFlow is usually already installed. We only need to install Keras Tuner and Optuna.

> If you are working locally and TensorFlow is not installed, install it in your environment first.


In [ ]:

!pip -q install keras-tuner optuna


## 2. Imports and Reproducibility

In [ ]:

import os
import gc
import time
import random
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import keras_tuner as kt
import optuna

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Keep the notebook fast for classroom demonstrations.
# Change to False for a broader, slower search.
FAST_MODE = True

MAX_EPOCHS = 25 if FAST_MODE else 80
RANDOM_TRIALS = 8 if FAST_MODE else 25
KT_TRIALS = 8 if FAST_MODE else 25
OPTUNA_TRIALS = 10 if FAST_MODE else 30

print("TensorFlow:", tf.__version__)
print("Keras Tuner:", kt.__version__)
print("Optuna:", optuna.__version__)



## 3. Dataset and Train / Validation / Test Split

We will use the **Breast Cancer Wisconsin** dataset because it is small and fast enough for classroom experimentation.

We split the data into:

- **Train**: used to update model weights.
- **Validation**: used to select hyperparameters.
- **Test**: reserved for the final model evaluation only.

> Do not use the test set while selecting hyperparameters.


In [ ]:

data = load_breast_cancer()

X = data.data
y = data.target

# 20% final test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

# From the remaining 80%, use 25% as validation => 60/20/20 overall
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25,
    stratify=y_train_val,
    random_state=SEED
)

# Fit scaler ONLY on training data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)



## 4. A Reusable Model-Building Function

To make the comparison fair, we will use the same model-building function for all experiments.

We will allow the following hyperparameters to change:

- `units1`
- `units2`
- `dropout`
- `learning_rate`
- `optimizer`
- `batch_size` during training


In [ ]:

def build_model(
    units1=64,
    units2=32,
    dropout=0.2,
    learning_rate=1e-3,
    optimizer_name="adam"
):
    tf.keras.backend.clear_session()

    model = keras.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(units1, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(units2, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(1, activation="sigmoid")
    ])

    if optimizer_name == "adam":
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == "rmsprop":
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        raise ValueError("optimizer_name must be 'adam' or 'rmsprop'")

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


def make_early_stopping():
    return keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )



### Helper: Train One Trial

Each **trial** represents one hyperparameter configuration.

Example trial:

```text
units1 = 64
units2 = 32
dropout = 0.2
learning_rate = 0.001
batch_size = 32
optimizer = Adam
```

The following function trains one model and returns the best validation loss and validation accuracy.


In [ ]:

def run_trial(config, epochs=MAX_EPOCHS, verbose=0):
    model = build_model(
        units1=config["units1"],
        units2=config["units2"],
        dropout=config["dropout"],
        learning_rate=config["learning_rate"],
        optimizer_name=config["optimizer"]
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=config["batch_size"],
        callbacks=[make_early_stopping()],
        verbose=verbose
    )

    best_val_loss = float(np.min(history.history["val_loss"]))
    best_val_accuracy = float(np.max(history.history["val_accuracy"]))

    del model
    gc.collect()

    return best_val_loss, best_val_accuracy


## 5. Baseline Model


Before tuning, we start with a **baseline model** using reasonable default values.

This is important because we first want to confirm that:

- the data pipeline is correct,
- the model trains successfully,
- the loss is decreasing,
- and the model is able to learn.

Only after that do we begin hyperparameter search.


In [ ]:

baseline_config = {
    "units1": 64,
    "units2": 32,
    "dropout": 0.2,
    "learning_rate": 1e-3,
    "batch_size": 32,
    "optimizer": "adam"
}

start = time.perf_counter()
baseline_loss, baseline_acc = run_trial(baseline_config, verbose=0)
baseline_time = time.perf_counter() - start

print("Baseline config:", baseline_config)
print(f"Best validation loss: {baseline_loss:.4f}")
print(f"Best validation accuracy: {baseline_acc:.4f}")
print(f"Time: {baseline_time:.1f} sec")



# 6. Grid Search

### Idea
We define a **fixed list of values** for each hyperparameter, then try **every possible combination**.

Here, we implement Grid Search directly in Python so we can clearly see what it is doing.

If we have:

- 2 values for `units1`
- 2 values for `units2`
- 2 learning rates
- 2 batch sizes

then the number of trials is:

`2 × 2 × 2 × 2 = 16 trials`

Each trial builds a new model, trains it, and records its validation performance.


In [ ]:

grid_space = {
    "units1": [32, 64],
    "units2": [16, 32],
    "learning_rate": [1e-3, 1e-4],
    "batch_size": [16, 32],
}

grid_results = []

start = time.perf_counter()

for units1, units2, lr, batch_size in itertools.product(
    grid_space["units1"],
    grid_space["units2"],
    grid_space["learning_rate"],
    grid_space["batch_size"]
):
    config = {
        "units1": units1,
        "units2": units2,
        "dropout": 0.2,          # fixed in this small grid
        "learning_rate": lr,
        "batch_size": batch_size,
        "optimizer": "adam"      # fixed in this small grid
    }

    val_loss, val_acc = run_trial(config)

    grid_results.append({
        **config,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

grid_time = time.perf_counter() - start

grid_df = pd.DataFrame(grid_results).sort_values("val_loss")
grid_df.head()


In [ ]:

grid_best_row = grid_df.iloc[0]

grid_best_config = {
    "units1": int(grid_best_row["units1"]),
    "units2": int(grid_best_row["units2"]),
    "dropout": float(grid_best_row["dropout"]),
    "learning_rate": float(grid_best_row["learning_rate"]),
    "batch_size": int(grid_best_row["batch_size"]),
    "optimizer": grid_best_row["optimizer"]
}

grid_best_loss = float(grid_best_row["val_loss"])
grid_best_acc = float(grid_best_row["val_accuracy"])

print("Best Grid Search configuration:")
print(grid_best_config)
print(f"Best validation loss: {grid_best_loss:.4f}")
print(f"Time: {grid_time:.1f} sec")



# 7. Random Search

### Idea
We define a larger search space, but we do not evaluate every possible combination.

Instead, we specify a fixed number of random trials, for example:

> Run 8 random trials.

Each trial samples one random hyperparameter combination.

This can greatly reduce the number of models we need to train compared with Grid Search.


In [ ]:

random_space = {
    "units1": [32, 64, 128],
    "units2": [16, 32, 64],
    "dropout": [0.0, 0.2, 0.4],
    "learning_rate": [1e-2, 1e-3, 1e-4],
    "batch_size": [16, 32, 64],
    "optimizer": ["adam", "rmsprop"],
}

all_possible = list(itertools.product(
    random_space["units1"],
    random_space["units2"],
    random_space["dropout"],
    random_space["learning_rate"],
    random_space["batch_size"],
    random_space["optimizer"],
))

rng = random.Random(SEED)
sampled_combinations = rng.sample(
    all_possible,
    k=min(RANDOM_TRIALS, len(all_possible))
)

random_results = []

start = time.perf_counter()

for units1, units2, dropout, lr, batch_size, optimizer in sampled_combinations:
    config = {
        "units1": units1,
        "units2": units2,
        "dropout": dropout,
        "learning_rate": lr,
        "batch_size": batch_size,
        "optimizer": optimizer
    }

    val_loss, val_acc = run_trial(config)

    random_results.append({
        **config,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

random_time = time.perf_counter() - start

random_df = pd.DataFrame(random_results).sort_values("val_loss")
random_df.head()


In [ ]:

random_best_row = random_df.iloc[0]

random_best_config = {
    "units1": int(random_best_row["units1"]),
    "units2": int(random_best_row["units2"]),
    "dropout": float(random_best_row["dropout"]),
    "learning_rate": float(random_best_row["learning_rate"]),
    "batch_size": int(random_best_row["batch_size"]),
    "optimizer": random_best_row["optimizer"]
}

random_best_loss = float(random_best_row["val_loss"])
random_best_acc = float(random_best_row["val_accuracy"])

print("Best Random Search configuration:")
print(random_best_config)
print(f"Best validation loss: {random_best_loss:.4f}")
print(f"Time: {random_time:.1f} sec")



# 8. Keras Tuner

**Keras Tuner is a tool, not a search strategy.**

We will define the search space once using a `HyperModel`.

Then we can use the same HyperModel with:

- `RandomSearch`
- `BayesianOptimization`
- `Hyperband`

We will also tune `batch_size` inside the `fit()` method.


In [ ]:

class TabularHyperModel(kt.HyperModel):

    def build(self, hp):
        tf.keras.backend.clear_session()

        units1 = hp.Choice("units1", [32, 64, 128])
        units2 = hp.Choice("units2", [16, 32, 64])
        dropout = hp.Choice("dropout", [0.0, 0.2, 0.4])

        learning_rate = hp.Choice(
            "learning_rate",
            [1e-2, 1e-3, 1e-4]
        )

        optimizer_name = hp.Choice(
            "optimizer",
            ["adam", "rmsprop"]
        )

        model = keras.Sequential([
            layers.Input(shape=(X_train.shape[1],)),
            layers.Dense(units1, activation="relu"),
            layers.Dropout(dropout),
            layers.Dense(units2, activation="relu"),
            layers.Dropout(dropout),
            layers.Dense(1, activation="sigmoid")
        ])

        if optimizer_name == "adam":
            optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
        else:
            optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)

        model.compile(
            optimizer=optimizer,
            loss="binary_crossentropy",
            metrics=["accuracy"]
        )

        return model

    def fit(self, hp, model, *args, **kwargs):
        # Training hyperparameter
        kwargs["batch_size"] = hp.Choice(
            "batch_size",
            [16, 32, 64]
        )

        return model.fit(*args, **kwargs)


hypermodel = TabularHyperModel()



## 8.1 Keras Tuner — RandomSearch

This is the same Random Search idea as before, but now Keras Tuner handles the process automatically:

1. Select a trial.
2. Build a model.
3. Train the model.
4. Measure `val_loss`.
5. Record the result.
6. Move automatically to the next trial.


In [ ]:

kt_random = kt.RandomSearch(
    hypermodel=hypermodel,
    objective="val_loss",
    max_trials=KT_TRIALS,
    seed=SEED,
    overwrite=True,
    directory="kt_results",
    project_name="random_search"
)

start = time.perf_counter()

kt_random.search(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=MAX_EPOCHS,
    callbacks=[make_early_stopping()],
    verbose=0
)

kt_random_time = time.perf_counter() - start

kt_random_best_hp = kt_random.get_best_hyperparameters(1)[0]
kt_random_best_trial = kt_random.oracle.get_best_trials(1)[0]

kt_random_best_config = dict(kt_random_best_hp.values)
kt_random_best_loss = float(kt_random_best_trial.score)

print("Best Keras Tuner RandomSearch config:")
print(kt_random_best_config)
print(f"Best validation loss: {kt_random_best_loss:.4f}")
print(f"Time: {kt_random_time:.1f} sec")



# 9. Bayesian Optimization — Keras Tuner

### Idea
Bayesian Optimization does not choose every trial independently.

After previous trials have been completed, it uses their results to build a model of the objective function and then selects a new trial that looks promising.

In Keras Tuner, `BayesianOptimization` uses a **Gaussian Process**-based approach.

The search space is the same; what changes is **how the next trial is selected**.


In [ ]:

bayes_tuner = kt.BayesianOptimization(
    hypermodel=hypermodel,
    objective="val_loss",
    max_trials=KT_TRIALS,
    seed=SEED,
    overwrite=True,
    directory="kt_results",
    project_name="bayesian_optimization"
)

start = time.perf_counter()

bayes_tuner.search(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=MAX_EPOCHS,
    callbacks=[make_early_stopping()],
    verbose=0
)

bayes_time = time.perf_counter() - start

bayes_best_hp = bayes_tuner.get_best_hyperparameters(1)[0]
bayes_best_trial = bayes_tuner.oracle.get_best_trials(1)[0]

bayes_best_config = dict(bayes_best_hp.values)
bayes_best_loss = float(bayes_best_trial.score)

print("Best Bayesian Optimization config:")
print(bayes_best_config)
print(f"Best validation loss: {bayes_best_loss:.4f}")
print(f"Time: {bayes_time:.1f} sec")



# 10. Hyperband / Successive Halving — Keras Tuner

### Idea
Hyperband does not give every trial the same training budget.

Instead of training every model to completion:

1. It starts with multiple configurations.
2. It gives them limited resources, such as a small number of epochs.
3. It stops weak configurations.
4. It gives more epochs to the promising trials.

This is especially useful when model training is expensive.


In [ ]:

hyperband_tuner = kt.Hyperband(
    hypermodel=hypermodel,
    objective="val_loss",
    max_epochs=MAX_EPOCHS,
    factor=3,
    seed=SEED,
    overwrite=True,
    directory="kt_results",
    project_name="hyperband"
)

start = time.perf_counter()

hyperband_tuner.search(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    callbacks=[make_early_stopping()],
    verbose=0
)

hyperband_time = time.perf_counter() - start

hyperband_best_hp = hyperband_tuner.get_best_hyperparameters(1)[0]
hyperband_best_trial = hyperband_tuner.oracle.get_best_trials(1)[0]

hyperband_best_config = dict(hyperband_best_hp.values)
hyperband_best_loss = float(hyperband_best_trial.score)

print("Best Hyperband config:")
print(hyperband_best_config)
print(f"Best validation loss: {hyperband_best_loss:.4f}")
print(f"Time: {hyperband_time:.1f} sec")



# 11. Optuna — TPE + Pruning

**Optuna is a tuning tool/library.**

In this example, we will use:

- **TPE Sampler** to select hyperparameters using a sequential model-based approach.
- **Median Pruner** to stop some unpromising trials early.

> This is not the same Gaussian Process-based Bayesian Optimization used in Keras Tuner.  
> Here, Optuna uses TPE as the sampler.


In [ ]:

class OptunaPruningCallback(keras.callbacks.Callback):
    def __init__(self, trial):
        super().__init__()
        self.trial = trial

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current_val_loss = logs.get("val_loss")

        if current_val_loss is None:
            return

        self.trial.report(float(current_val_loss), step=epoch)

        if self.trial.should_prune():
            raise optuna.TrialPruned()


def optuna_objective(trial):
    units1 = trial.suggest_categorical(
        "units1",
        [32, 64, 128]
    )

    units2 = trial.suggest_categorical(
        "units2",
        [16, 32, 64]
    )

    dropout = trial.suggest_float(
        "dropout",
        0.0,
        0.5,
        step=0.1
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        1e-2,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [16, 32, 64]
    )

    optimizer_name = trial.suggest_categorical(
        "optimizer",
        ["adam", "rmsprop"]
    )

    model = build_model(
        units1=units1,
        units2=units2,
        dropout=dropout,
        learning_rate=learning_rate,
        optimizer_name=optimizer_name
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        callbacks=[
            make_early_stopping(),
            OptunaPruningCallback(trial)
        ],
        verbose=0
    )

    best_val_loss = float(np.min(history.history["val_loss"]))

    del model
    gc.collect()

    return best_val_loss


In [ ]:

sampler = optuna.samplers.TPESampler(seed=SEED)

pruner = optuna.pruners.MedianPruner(
    n_startup_trials=3,
    n_warmup_steps=5
)

study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    pruner=pruner
)

start = time.perf_counter()

study.optimize(
    optuna_objective,
    n_trials=OPTUNA_TRIALS,
    show_progress_bar=True
)

optuna_time = time.perf_counter() - start

optuna_best_config = study.best_params
optuna_best_loss = float(study.best_value)

print("Best Optuna config:")
print(optuna_best_config)
print(f"Best validation loss: {optuna_best_loss:.4f}")
print(f"Time: {optuna_time:.1f} sec")



## 11.1 Inspect Optuna Trials

You may notice that some trials have the state:

- `COMPLETE`
- `PRUNED`

`PRUNED` means Optuna stopped the trial early because it was not promising.


In [ ]:

optuna_df = study.trials_dataframe()

display(
    optuna_df[
        [
            "number",
            "value",
            "state",
            "params_units1",
            "params_units2",
            "params_dropout",
            "params_learning_rate",
            "params_batch_size",
            "params_optimizer"
        ]
    ].sort_values("value", na_position="last")
)



# 12. Compare the Search Methods

We will compare the methods using the **best validation loss** reached by each method.

Important:

- This comparison is educational.
- Runtime depends on the number of trials, number of epochs, hardware, and search-space size.
- We still have not used the test set.


In [ ]:

comparison = pd.DataFrame([
    {
        "method": "Baseline",
        "best_val_loss": baseline_loss,
        "time_sec": baseline_time
    },
    {
        "method": "Grid Search",
        "best_val_loss": grid_best_loss,
        "time_sec": grid_time
    },
    {
        "method": "Random Search (manual)",
        "best_val_loss": random_best_loss,
        "time_sec": random_time
    },
    {
        "method": "Keras Tuner RandomSearch",
        "best_val_loss": kt_random_best_loss,
        "time_sec": kt_random_time
    },
    {
        "method": "Bayesian Optimization",
        "best_val_loss": bayes_best_loss,
        "time_sec": bayes_time
    },
    {
        "method": "Hyperband",
        "best_val_loss": hyperband_best_loss,
        "time_sec": hyperband_time
    },
    {
        "method": "Optuna TPE + Pruning",
        "best_val_loss": optuna_best_loss,
        "time_sec": optuna_time
    },
]).sort_values("best_val_loss")

comparison


In [ ]:

plt.figure(figsize=(10, 5))
plt.bar(comparison["method"], comparison["best_val_loss"])
plt.ylabel("Best Validation Loss")
plt.title("Hyperparameter Tuning Methods")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()



# 13. Select the Best Configuration Using Validation Performance

We will select the best configuration using validation loss only.

Then we will retrain a final model using Train + Validation and evaluate it on the Test Set **once at the end**.


In [ ]:

candidate_configs = [
    {
        "method": "Grid Search",
        "config": grid_best_config,
        "val_loss": grid_best_loss
    },
    {
        "method": "Random Search (manual)",
        "config": random_best_config,
        "val_loss": random_best_loss
    },
    {
        "method": "Keras Tuner RandomSearch",
        "config": kt_random_best_config,
        "val_loss": kt_random_best_loss
    },
    {
        "method": "Bayesian Optimization",
        "config": bayes_best_config,
        "val_loss": bayes_best_loss
    },
    {
        "method": "Hyperband",
        "config": hyperband_best_config,
        "val_loss": hyperband_best_loss
    },
    {
        "method": "Optuna TPE + Pruning",
        "config": optuna_best_config,
        "val_loss": optuna_best_loss
    }
]

winner = min(candidate_configs, key=lambda x: x["val_loss"])

print("Selected method:", winner["method"])
print("Selected validation loss:", winner["val_loss"])
print("Selected hyperparameters:")
print(winner["config"])



## 13.1 Normalize the Selected Configuration

Keras Tuner may add internal Hyperband keys such as `tuner/...`.

We will keep only the hyperparameters required by our model.


In [ ]:

def normalize_config(config):
    return {
        "units1": int(config["units1"]),
        "units2": int(config["units2"]),
        "dropout": float(config["dropout"]),
        "learning_rate": float(config["learning_rate"]),
        "batch_size": int(config["batch_size"]),
        "optimizer": str(config["optimizer"])
    }

final_config = normalize_config(winner["config"])
final_config



# 14. Final Training and ONE Test Evaluation

We have now finished selecting the hyperparameters.

We combine Train + Validation to retrain the final model, then use the Test Set for the final evaluation.

We will keep a small internal validation split only for Early Stopping during the final training stage.


In [ ]:

X_final_train = np.concatenate([X_train, X_val], axis=0)
y_final_train = np.concatenate([y_train, y_val], axis=0)

final_model = build_model(
    units1=final_config["units1"],
    units2=final_config["units2"],
    dropout=final_config["dropout"],
    learning_rate=final_config["learning_rate"],
    optimizer_name=final_config["optimizer"]
)

final_history = final_model.fit(
    X_final_train,
    y_final_train,
    validation_split=0.10,
    epochs=MAX_EPOCHS,
    batch_size=final_config["batch_size"],
    callbacks=[make_early_stopping()],
    verbose=0
)

test_loss, test_accuracy = final_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(f"Final Test Loss: {test_loss:.4f}")
print(f"Final Test Accuracy: {test_accuracy:.4f}")


In [ ]:

y_prob = final_model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(
    y_test,
    y_pred,
    target_names=data.target_names
))



# 15. What Should you Learn From This?

### Grid Search
- You define a fixed grid.
- The code evaluates every combination.
- Best suited for small search spaces.

### Random Search
- You define the search space and the number of trials.
- The code samples random combinations.
- More practical when the search space becomes large.

### Bayesian Optimization
- Each new trial uses information from previous trials.
- The goal is to choose more promising trials rather than sampling blindly.

### Hyperband
- Not every trial receives the same resources.
- Weak trials are stopped early.
- Useful when training epochs are expensive.

### Keras Tuner
- A tool that implements strategies such as RandomSearch, BayesianOptimization, and Hyperband.

### Optuna
- A flexible optimization library.
- In this notebook, we use TPE for sampling and MedianPruner for early stopping of weak trials.

---

## Practical Workflow

```text
Build Baseline
      ↓
Define Hyperparameters
      ↓
Define Search Space
      ↓
Choose Search Strategy
      ↓
Run Trials
      ↓
Compare Validation Performance
      ↓
Select Best Configuration
      ↓
Final Test Evaluation
```



# 16. Student Exercise

Modify the notebook and try the following:

1. Add `256` to the neuron choices.
2. Add `0.3` to the dropout choices.
3. Use Optuna to search a continuous learning-rate range:
   `1e-5 → 1e-2`.
4. Compare Random Search and Bayesian Optimization using the same number of trials.
5. Increase `MAX_EPOCHS` and observe the effect of Hyperband.
6. Change the objective from `val_loss` to `val_accuracy` and discuss the difference.
7. Count how many Optuna trials were pruned.



# 17. Key Takeaway

> **Hyperparameter tuning is an automated experiment loop.**

We define:

1. **What to tune**
2. **Search space**
3. **Search strategy**
4. **Objective metric**

Then the tuning tool automatically repeats:

```text
Choose Hyperparameters
        ↓
Build Model
        ↓
Train Model
        ↓
Evaluate on Validation Set
        ↓
Record Result
        ↓
Choose Next Trial
```

The Test Set is used only after the selection process is complete.



## Official References

- Keras Tuner documentation: https://keras.io/keras_tuner/
- Keras Tuner — Getting Started: https://keras.io/keras_tuner/getting_started/
- Keras Tuner — Custom HyperModel / training hyperparameters: https://keras.io/keras_tuner/guides/custom_tuner/
- Optuna documentation: https://optuna.readthedocs.io/
- Scikit-learn model selection documentation: https://scikit-learn.org/stable/modules/grid_search.html
